# nb-01 · Retrieve  *(stage: RETRIEVAL)*

**Purpose.** Pull raw reservoir-storage responses from the three source APIs into
immutable `data/original/`, recording a sha256 manifest. Nothing here is cleaned —
this is the faithful, re-fetchable copy of what each API returned.

| | |
|---|---|
| **Inputs** | `data/lookups/sources.yaml` (endpoints), `data/lookups/reservoirs.csv` (station seed), `$CDSS_API_KEY` (optional) |
| **Outputs** | `data/original/<source>/current/*.json` (immutable) · `data/original/manifest.json` |
| **Preconditions** | `uv sync`. A live pull needs network; an **offline demo** runs on the bundled sample. |

> **Immutable originals.** Files under `data/original/` are never edited in place.
> Re-cleaning re-runs parsers; re-fetching is explicit (`force=True`).

In [1]:
# Make the `reservoir` package importable when running from notebooks/.
# Cleaner: `uv pip install -e ..` once, then this cell is a no-op.
import sys, pathlib
SRC = pathlib.Path.cwd().parent / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print('package path:', SRC)

package path: /Users/briankeegan/Documents/GitHub/co-environmental-data/pipelines/reservoir-storage/src


In [2]:
from reservoir import config, fetch
import pandas as pd, json
registry = config.get_sources()
list(registry)

['dwr_cdss', 'reclamation_rise', 'northern_water']

## 1. What *would* be fetched (no network)

`discover()` is cheap and offline — it builds the API request URLs without
downloading. Inspect them before pulling.

In [3]:
plan = []
for slug, src in registry.items():
    for art in src.discover():
        plan.append({'source': slug, 'reservoir': art.metadata.get('reservoir_name'),
                     'file': art.local_path.name, 'url': art.url[:90] + '…'})
plan_df = pd.DataFrame(plan)
print(f'{len(plan_df)} artifacts planned across {plan_df.source.nunique()} sources')
plan_df.head(20)

17 artifacts planned across 3 sources


,source,reservoir,file,url
0,dwr_cdss,Green Mountain Reservoir,GREEN MOUNTAIN_STORAGE.json,https://dwr.state.co.us/Rest/GET/api/v2/teleme...
1,dwr_cdss,Green Mountain Reservoir,GREEN MOUNTAIN_ELEV.json,https://dwr.state.co.us/Rest/GET/api/v2/teleme...
2,dwr_cdss,Dillon Reservoir,DILLON_STORAGE.json,https://dwr.state.co.us/Rest/GET/api/v2/teleme...
3,dwr_cdss,Dillon Reservoir,DILLON_ELEV.json,https://dwr.state.co.us/Rest/GET/api/v2/teleme...
4,dwr_cdss,Williams Fork Reservoir,WILLIAMS FORK_STORAGE.json,https://dwr.state.co.us/Rest/GET/api/v2/teleme...
5,dwr_cdss,Williams Fork Reservoir,WILLIAMS FORK_ELEV.json,https://dwr.state.co.us/Rest/GET/api/v2/teleme...
6,dwr_cdss,Chatfield Reservoir,CHATFIELD_STORAGE.json,https://dwr.state.co.us/Rest/GET/api/v2/teleme...
7,dwr_cdss,Chatfield Reservoir,CHATFIELD_ELEV.json,https://dwr.state.co.us/Rest/GET/api/v2/teleme...
8,reclamation_rise,Blue Mesa Reservoir,item_None.json,https://data.usbr.gov/rise/api/result?itemId=N...
9,reclamation_rise,Blue Mesa Reservoir,item_None.json,https://data.usbr.gov/rise/api/result?itemId=N...


## 2. Fetch

Flip `LIVE = True` to pull from the real APIs (network + the `⚠️ VERIFY` endpoint
details in `sources.yaml` must be confirmed first — see `docs/survey-notes.md`).
`LIVE = False` seeds the bundled **offline sample** so nb-02 → nb-04 run end-to-end
without network.

In [6]:
LIVE = True  # ← set True for a real pull once the VERIFY items are confirmed

if LIVE:
    fetched = fetch.fetch_all()              # network: discover + download + manifest
    print(f'fetched {len(fetched)} artifacts')
else:
    # Offline demo: write the test fixture to the first DWR/CDSS artifact's path.
    import shutil
    fx = config.PROJECT_DIR / 'tests' / 'fixtures' / 'dwr_cdss_storage_sample.json'
    art = next(registry['dwr_cdss'].discover())
    art.local_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(fx, art.local_path)
    print('offline demo seeded:', art.local_path.relative_to(config.PROJECT_DIR))

HTTPError: 404 Client Error: Not Found for url: https://dwr.state.co.us/Rest/GET/api/v2/telemetrystations/telemetrytimeseriesday/?format=json&abbrev=GREEN+MOUNTAIN&parameter=ELEV&startDate=1900-01-01&pageSize=50000

## 3. Validate — what landed in `data/original/`

In [5]:
files = sorted(p for p in config.ORIGINAL.rglob('*.json') if p.name != 'manifest.json')
for p in files:
    print(p.relative_to(config.ORIGINAL), f'({p.stat().st_size:,} bytes)')
assert files, 'No raw files — run the fetch cell (LIVE or offline demo).'
print('\nnext → nb-02-audit')

dwr_cdss/current/GREEN MOUNTAIN_STORAGE.json (480 bytes)

next → nb-02-audit
